In [1]:
import os
import pennylane as qml
from pennylane import numpy as np
import pandas as pd
import tensorflow as tf
import pennylane as qml

from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.densenet import preprocess_input
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder


In [3]:
CSV_PATH = "D:\quantum dataset\sample_labels.csv"
IMAGE_DIR = "D:\quantum dataset\images"
IMG_SIZE = (224, 224)
BATCH_SIZE = 16

df = pd.read_csv(CSV_PATH)

# Encode disease labels
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["Finding Labels"])

NUM_CLASSES = len(df["label"].unique())
print("Classes:", label_encoder.classes_)


<>:1: SyntaxWarning: invalid escape sequence '\q'
<>:2: SyntaxWarning: invalid escape sequence '\q'
<>:1: SyntaxWarning: invalid escape sequence '\q'
<>:2: SyntaxWarning: invalid escape sequence '\q'
C:\Users\nagul\AppData\Local\Temp\ipykernel_6432\2795024879.py:1: SyntaxWarning: invalid escape sequence '\q'
  CSV_PATH = "D:\quantum dataset\sample_labels.csv"
C:\Users\nagul\AppData\Local\Temp\ipykernel_6432\2795024879.py:2: SyntaxWarning: invalid escape sequence '\q'
  IMAGE_DIR = "D:\quantum dataset\images"


Classes: ['Atelectasis' 'Atelectasis|Cardiomegaly'
 'Atelectasis|Cardiomegaly|Consolidation'
 'Atelectasis|Cardiomegaly|Consolidation|Effusion|Infiltration|Mass|Pleural_Thickening'
 'Atelectasis|Cardiomegaly|Effusion'
 'Atelectasis|Cardiomegaly|Effusion|Fibrosis|Infiltration'
 'Atelectasis|Cardiomegaly|Effusion|Fibrosis|Nodule'
 'Atelectasis|Cardiomegaly|Effusion|Infiltration|Pleural_Thickening'
 'Atelectasis|Cardiomegaly|Effusion|Mass'
 'Atelectasis|Cardiomegaly|Infiltration' 'Atelectasis|Consolidation'
 'Atelectasis|Consolidation|Edema'
 'Atelectasis|Consolidation|Edema|Effusion|Infiltration'
 'Atelectasis|Consolidation|Edema|Infiltration|Pneumonia'
 'Atelectasis|Consolidation|Effusion'
 'Atelectasis|Consolidation|Effusion|Emphysema'
 'Atelectasis|Consolidation|Effusion|Emphysema|Nodule|Pneumothorax'
 'Atelectasis|Consolidation|Effusion|Fibrosis|Pleural_Thickening'
 'Atelectasis|Consolidation|Effusion|Infiltration'
 'Atelectasis|Consolidation|Effusion|Infiltration|Pneumonia'
 'Atelec

In [4]:
datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

train_gen = datagen.flow_from_dataframe(
    dataframe=df,
    directory=IMAGE_DIR,
    x_col="Image Index",
    y_col="label",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="raw",
    subset="training",
    shuffle=False
)

val_gen = datagen.flow_from_dataframe(
    dataframe=df,
    directory=IMAGE_DIR,
    x_col="Image Index",
    y_col="label",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="raw",
    subset="validation",
    shuffle=False
)

Found 4485 validated image filenames.
Found 1121 validated image filenames.


In [5]:
# Split multi-label strings
all_diseases = set()

for labels in df["Finding Labels"]:
    for d in labels.split("|"):
        all_diseases.add(d.strip())

all_diseases = sorted(all_diseases)
print(all_diseases)


['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'No Finding', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']


In [6]:
cnn = DenseNet121(
    weights="imagenet",
    include_top=False,
    pooling="avg",
    input_shape=(224, 224, 3)
)
cnn.trainable = False


In [7]:
def extract_features(generator, model):
    features = []
    labels = []

    for i in range(len(generator)):
        x, y = generator[i] 
        f = model.predict(x, verbose=0)
        features.append(f)
        labels.append(y)

    return np.vstack(features), np.hstack(labels)

X_train, y_train = extract_features(train_gen, cnn)
X_val, y_val = extract_features(val_gen, cnn)

print("Feature shape:", X_train.shape)


Feature shape: (4485, 1024)


In [9]:
N_QUBITS = 4  # Keep ≤ 6 for simulation
pca = PCA(n_components=N_QUBITS)

X_train_pca = pca.fit_transform(X_train)
X_val_pca = pca.transform(X_val)


In [10]:
dev = qml.device("default.qubit", wires=N_QUBITS)

@qml.qnode(dev)
def quantum_circuit(inputs, weights):
    qml.templates.AngleEmbedding(inputs, wires=range(N_QUBITS))
    qml.templates.BasicEntanglerLayers(weights, wires=range(N_QUBITS))
    return qml.expval(qml.PauliZ(0))


In [11]:

class QuantumClassifier:
    def __init__(self):# increased circuit expressiveness
        self.weights = np.random.uniform(-1, 1, (5, N_QUBITS))


    def predict(self, X):
        return np.array([quantum_circuit(x, self.weights) for x in X])

    def loss(self, X, y):
        preds = self.predict(X)
        return np.mean((preds - y) ** 2)

    def train(self, X, y, epochs=20, lr=0.05):
        opt = qml.AdamOptimizer(lr)

        for i in range(epochs):
            self.weights = opt.step(lambda w: self.loss(X, y), self.weights)
            print(f"Epoch {i+1}, Loss: {self.loss(X, y):.4f}")




In [13]:
#1 for 
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(-np.pi, np.pi))
X_train_pca = scaler.fit_transform(X_train_pca)
X_val_pca = scaler.transform(X_val_pca)


In [19]:
print(X_train_pca[:2])


[[ 0.7099097  -1.8100977   0.38418895 -1.1280961 ]
 [ 1.5887517   0.34927452  0.32308787 -0.28753874]]


In [20]:
#fix 2 quantum output variance
outputs = [quantum_circuit(x, qc.weights) for x in X_train_pca[:20]]
print(np.var(outputs))


0.07615958502881102


In [21]:
print("Label distribution:", np.unique(y_train_binary[:200], return_counts=True))
print("PCA range:", X_train_pca.min(), X_train_pca.max())
print("Quantum output variance:", np.var([quantum_circuit(x, qc.weights) for x in X_train_pca[:20]]))


Label distribution: (tensor([0., 1.], requires_grad=True), array([  6, 194]))
PCA range: -3.1415927 3.141593
Quantum output variance: 0.07615958502881102


In [31]:
def predict_new_image(image_path):
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=IMG_SIZE)
    img = tf.keras.preprocessing.image.img_to_array(img)
    img = preprocess_input(np.expand_dims(img, axis=0))

    features = cnn.predict(img)
    features_pca = pca.transform(features)

    pred = quantum_circuit(features_pca[0], qc.weights)
    label = int(pred > 0.5)

    return label_encoder.inverse_transform([label])[0]


In [32]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(-np.pi, np.pi))
X_train_pca = scaler.fit_transform(X_train_pca)
X_val_pca = scaler.transform(X_val_pca)


In [33]:
#12 t1
# Collect unique diseases
all_diseases = set()

for labels in df["Finding Labels"]:
    for d in labels.split("|"):
        all_diseases.add(d.strip())

# Remove normal class
if "No Finding" in all_diseases:
    all_diseases.remove("No Finding")

all_diseases = sorted(all_diseases)

print("Diseases found:", all_diseases)
print("Number of diseases:", len(all_diseases))


Diseases found: ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']
Number of diseases: 14


In [22]:
# t2
import pennylane as qml
from pennylane import numpy as np
def create_binary_labels(df, disease):
    return df["Finding Labels"].apply(
        lambda x: 1.0 if disease in x else 0.0
    ).values


def balance_data(X, y, max_samples=100):
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        return None, None

    n = min(len(pos_idx), len(neg_idx), max_samples)

    np.random.shuffle(pos_idx)
    np.random.shuffle(neg_idx)

    idx = np.concatenate([pos_idx[:n], neg_idx[:n]])
    np.random.shuffle(idx)

    return X[idx], y[idx]


In [23]:
#t3 error to be implementeed before t2
from sklearn.model_selection import train_test_split

indices = np.arange(len(df))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

X_train_pca = X_pca[train_idx]
X_val_pca   = X_pca[val_idx]

df_train = df.iloc[train_idx].reset_index(drop=True)
df_val   = df.iloc[val_idx].reset_index(drop=True)


In [14]:
# generating X-features
#X_features.shape == (5606, 1024)

####--222

import os
import pennylane as qml
from pennylane import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tqdm import tqdm

IMAGE_DIR = r"D:\quantum dataset\images"   # change if needed
IMG_SIZE = (224, 224)
BATCH_SIZE = 16

####-3333

cnn = DenseNet121(
    weights="imagenet",
    include_top=False,
    pooling="avg",           # IMPORTANT → gives 1024 features
    input_shape=(224, 224, 3)
)

cnn.trainable = False

#####---4444

def load_and_preprocess_image(img_path):
    try:
        img = load_img(img_path, target_size=IMG_SIZE)
        img = img_to_array(img)
        img = preprocess_input(img)
        return img
    except Exception as e:
        return None
        
#####---555

features = []
valid_indices = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    img_name = row["Image Index"]     # change column name if needed
    img_path = os.path.join(IMAGE_DIR, img_name)

    if not os.path.exists(img_path):
        continue

    img = load_and_preprocess_image(img_path)
    if img is None:
        continue

    img = np.expand_dims(img, axis=0)   # shape (1, 224, 224, 3)

    feat = cnn.predict(img, verbose=0)  # shape (1, 1024)
    features.append(feat[0])
    valid_indices.append(idx)
    
######-666

X_features = np.array(features)

print("X_features shape:", X_features.shape)

###-777

df = df.iloc[valid_indices].reset_index(drop=True)

print("Updated df shape:", df.shape)

len(df) == len(X_features)

####-888

np.save("X_features.npy", X_features)
df.to_csv("metadata_clean.csv", index=False)
###-8.5
X_features = np.load("X_features.npy")
df = pd.read_csv("metadata_clean.csv")


100%|██████████████████████████████████████████████████████████████████████████████| 5606/5606 [42:58<00:00,  2.17it/s]


X_features shape: (5606, 1024)
Updated df shape: (5606, 12)


In [15]:
#from 'X_pca' is not defined
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
import pennylane as qml
from pennylane import numpy as np
N_QUBITS = 4

# PCA on ALL samples
pca = PCA(n_components=N_QUBITS)
X_pca = pca.fit_transform(X_features)

# Scale for quantum angles
scaler = MinMaxScaler(feature_range=(-np.pi, np.pi))
X_pca = scaler.fit_transform(X_pca)

print("X_pca shape:", X_pca.shape)


X_pca shape: (5606, 4)


In [28]:
from sklearn.model_selection import train_test_split

indices = np.arange(len(df))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

X_train_pca = X_pca[train_idx]
X_val_pca   = X_pca[val_idx]

df_train = df.iloc[train_idx].reset_index(drop=True)
df_val   = df.iloc[val_idx].reset_index(drop=True)

print(X_train_pca.shape, X_val_pca.shape)
print(len(df_train), len(df_val))


(4484, 4) (1122, 4)
4484 1122


In [35]:
y_binary = create_binary_labels(df_train, disease)
X_bal, y_bal = balance_data(X_train_pca, y_binary, max_samples=50)


In [34]:
print(X_pca.shape)
print(X_train_pca.shape)
print(df_train.shape)

(5606, 4)
(4484, 4)
(4484, 12)


In [20]:
#check the path for the pca and in it check for the path of images are not loading


In [24]:
import pennylane as qml
from pennylane import numpy as np   # 🔥 NOT normal numpy
N_QUBITS = 4
dev = qml.device("default.qubit", wires=N_QUBITS)

@qml.qnode(dev, diff_method="parameter-shift")
def quantum_circuit(x, weights):
    # Encode features
    for i in range(N_QUBITS):
        qml.RY(x[i], wires=i)

    # Trainable parameters
    for i in range(N_QUBITS):
        qml.RX(weights[i], wires=i)

    # Entanglement
    for i in range(N_QUBITS - 1):
        qml.CNOT(wires=[i, i + 1])

    return qml.expval(qml.PauliZ(0))
class QuantumClassifier:
    def __init__(self):
        self.weights = np.random.uniform(
            -0.5, 0.5,
            size=(N_QUBITS,),
            requires_grad=True   # 🔥 MANDATORY
        )

    def loss(self, weights, X, y):
        preds = np.array([
            quantum_circuit(x, weights) for x in X
        ])
        preds = (preds + 1) / 2
        eps = 1e-8
        return -np.mean(
            y * np.log(preds + eps) +
            (1 - y) * np.log(1 - preds + eps)
        )

    def train(self, X, y, epochs=30, lr=0.02):
        opt = qml.AdamOptimizer(lr)

        for epoch in range(epochs):
            self.weights = opt.step(
                lambda w: self.loss(w, X, y),
                self.weights
            )

            current_loss = self.loss(self.weights, X, y)
            print(f"Epoch {epoch+1}, Loss: {current_loss:.4f}")
from pennylane import grad

qc = QuantumClassifier()

g = grad(lambda w: quantum_circuit(X_train_pca[0], w))(qc.weights)
print("Gradient norm:", np.linalg.norm(g))
quantum_models = {}

for disease in all_diseases:
    print(f"\nTraining disease: {disease}")

    y_binary = create_binary_labels(df_train, disease)

    X_bal, y_bal = balance_data(
        X_train_pca,
        y_binary,
        max_samples=50
    )

    if X_bal is None:
        print(f"⚠️ Skipping {disease}")
        continue

    print("Balanced labels:", np.unique(y_bal, return_counts=True))

    qc = QuantumClassifier()
    qc.train(X_bal, y_bal, epochs=30, lr=0.02)

    quantum_models[disease] = qc


Gradient norm: 0.22441614060574894

Training disease: Atelectasis
Balanced labels: (array([0., 1.]), array([50, 50]))
Epoch 1, Loss: 1.0928
Epoch 2, Loss: 1.0804
Epoch 3, Loss: 1.0682
Epoch 4, Loss: 1.0561
Epoch 5, Loss: 1.0442
Epoch 6, Loss: 1.0326
Epoch 7, Loss: 1.0211
Epoch 8, Loss: 1.0099
Epoch 9, Loss: 0.9989
Epoch 10, Loss: 0.9881
Epoch 11, Loss: 0.9775
Epoch 12, Loss: 0.9672
Epoch 13, Loss: 0.9571
Epoch 14, Loss: 0.9473
Epoch 15, Loss: 0.9376
Epoch 16, Loss: 0.9282
Epoch 17, Loss: 0.9191
Epoch 18, Loss: 0.9102
Epoch 19, Loss: 0.9015
Epoch 20, Loss: 0.8930
Epoch 21, Loss: 0.8848
Epoch 22, Loss: 0.8768
Epoch 23, Loss: 0.8691
Epoch 24, Loss: 0.8616
Epoch 25, Loss: 0.8543
Epoch 26, Loss: 0.8472
Epoch 27, Loss: 0.8403
Epoch 28, Loss: 0.8337
Epoch 29, Loss: 0.8273
Epoch 30, Loss: 0.8210

Training disease: Cardiomegaly
Balanced labels: (array([0., 1.]), array([50, 50]))
Epoch 1, Loss: 1.1362
Epoch 2, Loss: 1.1199
Epoch 3, Loss: 1.1042
Epoch 4, Loss: 1.0890
Epoch 5, Loss: 1.0743
Epoch 6

In [36]:
#predicting new x-ray
#load image
import pickle

with open("quantum_models.pkl", "wb") as f:
    pickle.dump(quantum_models, f)
with open("pca.pkl", "wb") as f:
    pickle.dump(pca, f)

with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)



In [37]:
#predicting new x-ray
def predict_new_xray(image_path, quantum_models, threshold=0.5):
    # Load & preprocess
    img = load_and_preprocess_image(image_path)
    img = np.expand_dims(img, axis=0)

    # CNN features
    features = cnn.predict(img, verbose=0)

    # PCA transform
    x_pca = scaler.transform(pca.transform(features))[0]

    predictions = {}

    for disease, qc in quantum_models.items():
        score = quantum_circuit(x_pca, qc.weights)
        prob = (score + 1) / 2
        predictions[disease] = float(prob)

    detected = [d for d, p in predictions.items() if p >= threshold]
    return detected, predictions


In [41]:
import pickle

with open("quantum_models.pkl", "wb") as f:
    pickle.dump(quantum_models, f)
with open("pca.pkl", "wb") as f:
    pickle.dump(pca, f)

with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)


In [28]:
def predict_new_xray(image_path, quantum_models, threshold=0.516):
    # Load & preprocess
    img = load_and_preprocess_image(image_path)
    img = np.expand_dims(img, axis=0)

    # CNN features
    features = cnn.predict(img, verbose=0)

    # PCA transform
    x_pca = scaler.transform(pca.transform(features))[0]

    predictions = {}

    for disease, qc in quantum_models.items():
        score = quantum_circuit(x_pca, qc.weights)
        prob = (score + 1) / 2
        predictions[disease] = float(prob)

    detected = [d for d, p in predictions.items() if p >= threshold]
    return detected, predictions


In [36]:
detected, probs = predict_new_xray(
    r"D:\quantum dataset\images\00000013_005.png",
    quantum_models,
    threshold=0.5156
)

print("Detected diseases:", detected)


Detected diseases: ['Infiltration']


In [39]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten


In [41]:
print("y_train shape:", y_train.shape)


y_train shape: (4485,)


In [44]:
import numpy as np

print("y_train shape:", np.array(y_train).shape)
print("y_train first 5:", y_train[:5])
print("unique values:", np.unique(y_train)[:20])


y_train shape: (4485,)
y_train first 5: [234 234 234 222 234]
unique values: [ 0  1  2  3  4  7  8 10 11 13 14 16 17 18 19 20 21 22 23 24]


In [49]:
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

base_model = DenseNet121(weights="imagenet", include_top=False, input_shape=(224,224,3))

x = GlobalAveragePooling2D()(base_model.output)
output = Dense(14, activation="sigmoid")(x)   # if multi-label

model = Model(inputs=base_model.input, outputs=output)


In [54]:
model.save("my_model.keras")


In [55]:
model = load_model("my_model.keras")


In [56]:
import os
print(os.listdir())


['.gitconfig', '.ipynb_checkpoints', '.ipython', '.jupyter', '.keras', '.lesshst', '.matplotlib', '.ms-ad', '.vscode', 'AppData', 'Application Data', 'Cookies', 'Local Settings', 'metadata_clean.csv', 'My Documents', 'my_model.keras', 'NetHood', 'NTUSER.DAT', 'ntuser.dat.LOG1', 'ntuser.dat.LOG2', 'NTUSER.DAT{566edc62-beb8-11ef-ad99-d019d827ebcb}.TM.blf', 'NTUSER.DAT{566edc62-beb8-11ef-ad99-d019d827ebcb}.TMContainer00000000000000000001.regtrans-ms', 'NTUSER.DAT{566edc62-beb8-11ef-ad99-d019d827ebcb}.TMContainer00000000000000000002.regtrans-ms', 'ntuser.ini', 'OneDrive', 'pca.pkl', 'PrintHood', 'quantum.ipynb', 'quantum_models.pkl', 'Recent', 'scaler.pkl', 'SendTo', 'Start Menu', 'Templates', 'Untitled.ipynb', 'Untitled1.ipynb', 'Untitled2.ipynb', 'Untitled3.ipynb', 'Untitled4.ipynb', 'X_features.npy']


In [57]:
from tensorflow.keras.models import load_model

model = load_model("my_model.keras")
print("✅ Model loaded successfully!")


✅ Model loaded successfully!


In [58]:
import os
from tensorflow.keras.models import load_model

print("Current folder:", os.getcwd())

model_path = os.path.join(os.getcwd(), "my_model.keras")
model = load_model(model_path)

print("✅ Model loaded successfully from:", model_path)


Current folder: C:\Users\nagul
✅ Model loaded successfully from: C:\Users\nagul\my_model.keras


In [69]:
import numpy as np
from tensorflow.keras.preprocessing import image

img_path = r"D:\quantum dataset\images\00000013_005.png" # change name

img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = img_array / 255.0

pred = model.predict(img_array)
print("Prediction:", pred)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 396ms/step
Prediction: [[0.81488407 0.6872239  0.14352988 0.83747554 0.49884933 0.15999582
  0.50375366 0.71207565 0.66005105 0.25930664 0.6330254  0.3901952
  0.6844958  0.27043203]]


In [70]:
pred = pred[0]
labels = (pred > 0.5).astype(int)
print("Labels:", labels)
print("Probabilities:", pred)


Labels: [1 1 0 1 0 0 1 1 1 0 1 0 1 0]
Probabilities: [0.81488407 0.6872239  0.14352988 0.83747554 0.49884933 0.15999582
 0.50375366 0.71207565 0.66005105 0.25930664 0.6330254  0.3901952
 0.6844958  0.27043203]
